# 00 · Tutorial: MIL-CREDA on EEG

A usage example of MIL-CREDA, the method of the paper, on EEG. It runs the paper's own code, unchanged, on

* **(a) one simulated unit** of the synthetic effect-by-shift study (class effect $d_{\mathrm{C}}=1.5$, shift 1.5,
  rotation 0: the unit of the paper's illustrations), from the 19-channel 10-20 source to the 14-channel EMOTIV
  target, with the simulator's ground truth to look at what the attention weighs;
* **(b) one cross-subject split** of the public ADHD/Control Children recordings (IEEE DataPort, Nasrabadi et al.),
  rotation 3, target-test half 1, read from the public Kaggle dataset `daprosero/mi-tdah-dataset` (CC0).

Each part does the two phases of the paper: **phase 1** fits a source model (the base every method starts from) and
**phase 2** adapts a target encoder with the MIL-CREDA objective, choosing its configuration without any target
label by **cef**, the effective number of predicted classes. This is a tutorial, not a paper stage: the stage
notebooks `01`-`06` regenerate the paper's results; this one shows how one run of the method works.

**Budget.** It runs on a laptop CPU in about 7 minutes (measured on an Apple M2, 4 threads). The paper's values are
kept except the ones in this table, which are reduced so the notebook stays short:

| what | here | paper |
|---|---|---|
| phase-1 search | none: refit once at the unit's recorded TPE settings | TPE, 20 trials x 3 inner splits |
| phase-2 pool, simulated unit | first 4 configurations of the paper's seeded draw | 100 configurations |
| phase-2 pool, IEEE split | first 22 configurations (the paper's cef pick is trial 21, with cef 2, the maximum for two classes, so the 78 later trials could not change the paper's pick) | 100 configurations |
| phase-2 epochs | 100 (unchanged) | 100 |

Set the environment variable `REPRO_SMOKE=1` for a 2-minute wiring check (1 and 2 configurations, 10 epochs, 24
evaluation subjects); its numbers are not meaningful.

## Setup
Works locally (repository checkout) and on Kaggle (the repository is cloned into `/kaggle/working`).

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/Daprosero/Domain_Adaptation"
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / "src" / "MIL_CREDA_EEG").is_dir()), None)
if root is None and Path("/kaggle").exists():
    root = Path("/kaggle/working/Domain_Adaptation")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mne==1.10.1", "neurodsp==2.2.1"], check=True)
if root is None:
    raise RuntimeError("run this notebook from inside the Domain_Adaptation repository or on Kaggle")
sys.path.insert(0, str(root / "src"))

import contextlib
import matplotlib.pyplot as plt
import numpy as np
import torch

from MIL_CREDA_EEG import repro
from MIL_CREDA_EEG.reproduce import tutorial as T

torch.set_num_threads(min(4, os.cpu_count() or 1))
STARTED = time.time()
print("repository:", root)

## MIL-CREDA in one page

Each subject is a **bag** of 4 s windows with one label; the windows carry none. Per domain $\delta\in\{s,t\}$ an
encoder $\mathcal F^{\delta}$ maps each window to $\mathbf h\in\mathbb R^{d}$ (here: absolute and relative log band
power per channel, 1-4, 4-8, 8-13, 13-30, 30-45 Hz, then a linear projection). The attention logit of window $a$ of
bag $u$ is a learned relevance plus a kernel consensus, and a softmax with temperature $\tau_{\mathrm{att}}$ gives the
weights:

$$\nu_{u,a}=\mathbf v_R^\top\tanh(\mathbf V_R\mathbf h_{u,a}+\mathbf b_R)+\tfrac{\gamma}{m_u-1}\sum_{a'\neq a}
\kappa_\sigma(\mathbf h_{u,a},\mathbf h_{u,a'}),\qquad
\beta_{u,a}=\operatorname{softmax}_a(\nu_{u,a}/\tau_{\mathrm{att}}).$$

The bag is $\Psi(B_u)=\sum_a\beta_{u,a}\Phi(\mathbf h_{u,a})$ in the RKHS of the Gaussian kernel, whose inner products
give the **bag kernel** $\kappa^B$; its Euclidean summary $\mathbf z_u=\sum_a\beta_{u,a}\mathbf h_{u,a}$ feeds a linear
softmax classifier. The cost is

$$\mathcal L=\mathcal L_{\mathrm{src}}+\eta_{\mathrm{glob}}\mathcal L_{\mathrm{glob}}+\eta_{\mathrm{loc}}\mathcal L_{\mathrm{loc}},$$

with $\mathcal L_{\mathrm{src}}$ the normalized cross-entropy on source bags; $\mathcal L_{\mathrm{glob}}$ the
class-conditional alignment of source and confidence-weighted, pseudo-labeled target bags through the quadratic
matrix-based Rényi entropy of their balanced joint bag-kernel matrix; and $\mathcal L_{\mathrm{loc}}$ the distance
of each target bag to its personalized source reference (a softmax over $\kappa^B/\tau_{\mathrm{loc}}$ of the source
bags of its pseudo-labeled class). All three lie in $[0,1]$.

**Phase 1** fits the source encoder, the attention and the classifier with $\mathcal L_{\mathrm{src}}$ ($\gamma=0$),
fixes $\sigma$ at the median distance between windows of different source bags, and fixes a decision cut on an inner
validation split. **Phase 2** starts the target encoder from the source one and trains it, with the attention, on
$\mathcal L$; the source encoder and the classifier stay frozen. No target label enters training or selection.

The code below calls `MIL_CREDA_EEG.reproduce.tutorial`, a thin layer over the paper's entrypoints:
`xpilot.build_sides` (data and subject-wise split), `xcompare.shared_base` (phase 1),
`criteria_pool.draw` / `criteria_pool.train` (the seeded phase-2 pool and one phase-2 run, with the learning rate
annealed as $\eta_2(1+10p)^{-0.75}$) and `phase2_select.cef_score` (the label-free criterion).

## Parameters

In [ ]:
SMOKE = os.environ.get("REPRO_SMOKE") == "1"
EPOCHS2 = 10 if SMOKE else T.PAPER_EPOCHS2               # paper: 100
SIM_TRIALS = 1 if SMOKE else 4                           # paper: 100
IEEE_TRIALS = 2 if SMOKE else T.ieee_pick()["trial"] + 1  # paper: 100; its cef pick is trial 21
EVAL_SUBJECTS = 24 if SMOKE else 120                     # paper: 120 fresh evaluation subjects

print("simulated unit:", T.SIM_UNIT["key"], T.SIM_UNIT["kwargs"])
print("  phase-1 settings (recorded TPE pick):", T.SIM_UNIT["phase1"])
print("IEEE split:", T.IEEE_UNIT["key"], "(rotation", T.IEEE_UNIT["rotation"], "half", str(T.IEEE_UNIT["half"]) + ")")
print("  phase-1 settings (recorded TPE pick):", T.IEEE_UNIT["phase1"])
print("phase-2 space of MIL-CREDA:", {k: (d.low, d.high, "log" if d.log else "lin")
                                      for k, d in T.C.space("MIL-CREDA").items()})
print(f"phase-2 epochs {EPOCHS2}, configurations: simulated {SIM_TRIALS}, IEEE {IEEE_TRIALS}, smoke={SMOKE}")

# (a) A simulated unit

## Data

`eeg_sim` draws each subject from a spherical head model with six radial dipoles emitting 1/f activity and theta,
alpha and beta bursts; in the ADHD class frontal theta rises and beta falls, calibrated so that Cohen's $d$ of the
theta/beta ratio at Fz is 1.5. Blinks and muscle bursts are added at 3 per minute, and 20 % of the windows of an
ADHD subject concentrate the class effect (*class events*). Source and target hold 60 subjects each (30 per class),
each a bag of 60 windows of 4 s at 128 Hz; the target is recorded by a 14-channel EMOTIV-like device whose gain,
noise and scale grow with the shift. The split is subject-wise: one fifth of the source for test, the target in
five folds (rotation 0 tests fold 0, 12 subjects, and adapts on the other 48 without labels). The
**evaluation pool** is 120 fresh target subjects that only the simulator labels.

In [ ]:
t0 = time.time()
SIM = T.sim_unit(EVAL_SUBJECTS)
sides = SIM["sides"]
print("unit:", SIM["unit"], f"({time.time() - t0:.0f} s to simulate)")
for role in ("source", "source_test", "target", "target_test"):
    print(f"  {role:12s} {len(sides[role]):3d} bags, windows x channels x samples = {tuple(sides[role][0].windows.shape)}")
print("  evaluation pool", len(SIM["pool"]), "subjects")
print("measured Cohen's d of the theta/beta ratio:", SIM["sim_record"]["measuredD"])

In [ ]:
from MIL_CREDA_EEG import eeg_sim as E
fig, axes = plt.subplots(1, 2, figsize=(12, 2.8), sharey=True)
t = np.arange(E.WINDOW) / E.RATE
for ax, role, names, ch in ((axes[0], "source", E.TEN_TWENTY, "Fz"), (axes[1], "target", E.EMOTIV, "F3")):
    for label, color in ((0, "tab:blue"), (1, "tab:red")):
        bag = next(b for b in sides[role] if b.label == label)
        ax.plot(t, bag.windows[0, list(names).index(ch)].numpy(), color=color, lw=0.7,
                label=("ADHD" if label else "control") + f" ({bag.subject})")
    ax.set_title(f"{role}: first window at {ch}")
    ax.set_xlabel("s")
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Phase 1: the source model

`xcompare.shared_base` fits phase 1 at the unit's settings: once on an inner split, to place the decision cut where
the subject-level balanced accuracy of the inner validation subjects peaks, and once on all 48 source training
subjects, the model every method adapts. Its bridged copy (target encoder initialized from the source one: the
projection columns of the shared channels copied, the others at zero) predicts the target **without adaptation**.

In [ ]:
stack = contextlib.ExitStack()
stack.enter_context(T.running(SIM))      # the spectral trunk and the light phase-2 trace, as the paper runs
t0 = time.time()
SIM_BASE = T.phase_one(SIM, T.SIM_UNIT["phase1"])
print(f"phase 1: {time.time() - t0:.0f} s, kernel width sigma = {SIM_BASE['base'].sigma:.4f}, "
      f"cut = {SIM_BASE['cut']:.4f} (paper base {T.SIM_UNIT['paperCut']:.4f})")
print("source test (12 subjects):", SIM_BASE["sourceTest"])
SIM_BEFORE = T.scores(SIM_BASE["prepared"], SIM["pool"], SIM_BASE["cut"])
print("target, no adaptation, evaluation pool:", SIM_BEFORE)
print(SIM_BASE["prepared"])

## Phase 2: a pool of configurations and the label-free pick

The paper trains 100 configurations per method drawn by a seeded random sampler over the phase-2 space (learning
rate, weight decay, $\eta_{\mathrm{glob}}$, $\eta_{\mathrm{loc}}$, $\tau_{\mathrm{loc}}$, $\gamma$, $\tau_{\mathrm{att}}$),
each for 100 epochs, and picks the one with the highest **cef** on the unlabeled target adaptation bags (first
maximum). Here the first few configurations of that same draw are trained. The evaluation-pool BA is printed for
reading only; it plays no part in the pick.

In [ ]:
rows = []
for trial in range(SIM_TRIALS):
    t0 = time.time()
    params = T.pool_params(trial)
    model = T.phase_two(SIM, SIM_BASE, params, T.C.train_seed(T.POOL_SEED, trial), EPOCHS2)
    rows.append({"trial": trial, "model": model, "cef": T.cef(model, sides["target"]),
                 "pool": T.scores(model, SIM["pool"], SIM_BASE["cut"]),
                 "test": T.scores(model, sides["target_test"], SIM_BASE["cut"])})
    r = rows[-1]
    print(f"trial {trial}: cef {r['cef']:.3f}  pool BA {r['pool']['BA']:.3f} AUC {r['pool']['AUC']:.3f}  "
          f"({time.time() - t0:.0f} s)  " + ", ".join(f"{k}={v:.3g}" for k, v in params.items()))
SIM_PICK = rows[T.cef_pick(rows)]
print(f"\ncef pick: trial {SIM_PICK['trial']}")
print(f"{'':28s} {'BA':>6s} {'AUC':>6s}")
print(f"{'no adaptation (phase 1)':28s} {SIM_BEFORE['BA']:6.3f} {SIM_BEFORE['AUC']:6.3f}")
print(f"{'MIL-CREDA, cef pick':28s} {SIM_PICK['pool']['BA']:6.3f} {SIM_PICK['pool']['AUC']:6.3f}")
pp = T.SIM_UNIT["paperPick"]
print(f"paper (100-configuration pool): cef pick trial {pp['trial']}, pool BA {pp['BApool']:.3f}, AUC {pp['AUCpool']:.3f}")

## What the attention weighs

The paper's illustration rule, fixed before reading any weight: the first evaluation subject with ADHD whose bag
holds class events and both artifact types. Its 60 window weights are shown after phase 1 and after adaptation,
with the simulator's masks (which never enter training). An AUC above 0.5 means less weight on artifact windows
(inverted weight against the artifact mask) or more weight on class-event windows; 0.5 is uniform weighting.

In [ ]:
subject = T.illustration_subject(SIM["pool"])
m = T.masks(subject)
w_before, w_after = T.attention(SIM_BASE["prepared"], subject), T.attention(SIM_PICK["model"], subject)
fig, ax = plt.subplots(figsize=(12, 3))
k = np.arange(len(w_after))
for j in k:
    if m["event"][j]:
        ax.axvspan(j - 0.5, j + 0.5, color="tab:green", alpha=0.15, lw=0)
    if m["artifact"][j]:
        ax.axvspan(j - 0.5, j + 0.5, color="tab:orange" if m["artifactType"][j] == "blink" else "tab:purple",
                   alpha=0.25, lw=0)
ax.step(k, w_before, where="mid", color="0.5", label="after phase 1")
ax.step(k, w_after, where="mid", color="k", label="after phase 2 (cef pick)")
ax.axhline(1 / len(k), color="0.7", ls=":", label="uniform")
ax.set_xlabel("window (2 s hop)")
ax.set_ylabel(r"attention $\beta$")
ax.set_title(f"{subject.subject}: green = class events, orange = blinks, purple = muscle")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
for name, w in (("after phase 1", w_before), ("after phase 2", w_after)):
    a = T.mask_aucs(w, subject)
    print(f"{name}: artifact AUC {a['artifactAUC']:.3f}, class-event AUC {a['eventAUC']:.3f}")

# (b) A cross-subject split of ADHD/Control Children

## Data

`repro.ieee_root()` finds the 121 recordings (61 ADHD, 60 controls; 19 channels at 128 Hz): mounted when the Kaggle
dataset is attached, otherwise downloaded (about 33 MB) with `kagglehub`. Each recording is notch-filtered at 50 Hz
and cut into 4 s windows with 50 % overlap, at most 60 per subject, each band-passed between 1 and 45 Hz and
re-referenced to the common average. Partition, subject-wise and stratified: five folds; rotation 3 takes fold 3 as
the target, split into an unlabeled adaptation half and a test half (half 1 tests the odd-indexed subjects of each class), fold 4 is the source
test and the other three train.

In [ ]:
t0 = time.time()
IEEE = T.ieee_unit()
isides = IEEE["sides"]
print("data:", IEEE["root"], f"({time.time() - t0:.0f} s to read)")
for role in ("source", "source_test", "target", "target_test"):
    y = [b.label for b in isides[role]]
    print(f"  {role:12s} {len(y):3d} subjects ({sum(y)} ADHD), first bag {tuple(isides[role][0].windows.shape)}")

## Phase 1

In [ ]:
t0 = time.time()
IEEE_BASE = T.phase_one(IEEE, T.IEEE_UNIT["phase1"])
print(f"phase 1: {time.time() - t0:.0f} s, sigma = {IEEE_BASE['base'].sigma:.4f}, cut = {IEEE_BASE['cut']:.4f} "
      f"(paper base {T.IEEE_UNIT['paperCut']:.4f})")
print("source test:", IEEE_BASE["sourceTest"])
IEEE_BEFORE = T.scores(IEEE_BASE["prepared"], isides["target_test"], IEEE_BASE["cut"])
print("target test, no adaptation:", IEEE_BEFORE)

## Phase 2 and the cef pick

The same seeded draw as in the paper's ADHD pool (seed 1), the same training seeds, 100 epochs. The test half is
read only to report; cef reads the predictions on the 12 unlabeled adaptation subjects. Next to each cef this run
measures is the cef the paper's run measured for the same configuration (`tutorial.IEEE_UNIT["paperCef"]`).

In [ ]:
irows = []
paper_cef = T.IEEE_UNIT["paperCef"]
for trial in range(IEEE_TRIALS):
    t0 = time.time()
    model = T.phase_two(IEEE, IEEE_BASE, T.pool_params(trial), T.C.train_seed(T.POOL_SEED, trial), EPOCHS2)
    irows.append({"trial": trial, "model": model, "cef": T.cef(model, isides["target"]),
                  "test": T.scores(model, isides["target_test"], IEEE_BASE["cut"])})
    r = irows[-1]
    print(f"trial {trial:2d}: cef {r['cef']:.3f} (paper {paper_cef[trial]:.3f})  "
          f"test BA {r['test']['BA']:.3f} AUC {r['test']['AUC']:.3f}  ({time.time() - t0:.0f} s)")
IEEE_PICK = irows[T.cef_pick(irows)]
oracle = max(irows, key=lambda r: r["test"]["BA"])
print(f"\ncef pick of this run: trial {IEEE_PICK['trial']}; cef pick of the paper's cef values: trial "
      f"{T.cef_pick([{'cef': c} for c in paper_cef[:IEEE_TRIALS]])}; "
      f"oracle among these (reads test labels): trial {oracle['trial']}")
print(f"{'':28s} {'BA':>6s} {'AUC':>6s} {'kappa':>6s}")
for name, s in (("no adaptation (phase 1)", IEEE_BEFORE), ("MIL-CREDA, cef pick", IEEE_PICK["test"])):
    print(f"{name:28s} {s['BA']:6.3f} {s['AUC']:6.3f} {s['kappa']:6.3f}")

**Reading the comparison.** Phase 2 is deterministic for a given machine, but its floating-point result can differ
in the last bits between machines (CPU, BLAS). Most configurations land on the same predictions, so their cef
matches the paper's to the printed digits; a configuration with a large learning rate (trial 2 has
$\eta_2\approx 0.009$, the top of the range) can flip one adaptation subject and change its cef, and with it the
first maximum. The cell below trains nothing: it reads the paper's pick (trial 21,
`Manifests/attention_influence/ieee_cef_picks.json`) from the runs above and compares its test BA, AUC and kappa
with the paper's.

In [ ]:
paper = T.ieee_pick()
if paper["trial"] < len(irows):
    got = irows[paper["trial"]]["test"]
    repro.compare([("test BA, paper's pick", got["BA"], paper["BA"], 0.02),
                   ("test AUC, paper's pick", got["AUC"], paper["AUC"], 0.02),
                   ("test kappa, paper's pick", got["kappa"], paper["kappa"], 0.05),
                   ("cef, paper's pick", irows[paper["trial"]]["cef"], paper["cef"], 1e-3)]);
else:
    print(f"the paper's pick is trial {paper['trial']}; this run trained {len(irows)} configurations (smoke mode)")

## Attention over a test subject

ADHD/Control Children carries no window masks, so the weights are shown as they are: the 60 windows of the first
ADHD test subject, after phase 1 and after adaptation.

In [ ]:
subject = next(b for b in isides["target_test"] if b.label == 1)
fig, ax = plt.subplots(figsize=(12, 2.8))
for model, name, color in ((IEEE_BASE["prepared"], "after phase 1", "0.5"), (IEEE_PICK["model"], "after phase 2", "k")):
    w = T.attention(model, subject)
    ax.step(np.arange(len(w)), w, where="mid", color=color, label=name)
ax.axhline(1 / len(w), color="0.7", ls=":", label="uniform")
ax.set_xlabel("window (2 s hop)")
ax.set_ylabel(r"attention $\beta$")
ax.set_title(subject.subject)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
stack.close()
print(f"total runtime: {(time.time() - STARTED) / 60:.1f} min")

## Where to go next

* `02_Synthetic_Criteria_Pool` trains the full 100-configuration pools of every method on the 24 synthetic units
  and compares the label-free criteria; `04_ADHD_Pool` does the same on the ten ADHD units.
* `03_Synthetic_Mechanisms` and `05_Attention_Influence` measure what the attention and the local term do over all
  units; `06_ADHD_Graph_Analysis` builds attention-weighted connectivity graphs.
* Reproducibility. The simulated subjects are drawn afresh, and the simulator's floating-point output differs
  slightly between platforms (the AUC of the unadapted target on the evaluation pool is 0.896 on an Apple M2 and
  0.897 in the paper's Kaggle run), so a configuration can land on the other side of a collapse: the paper's cef
  pick for this unit, trial 34, predicts a single class on the M2. The ADHD recordings are fixed and their
  phase 1 refits to the paper's cut; there only the high-learning-rate trial 2 differs, as shown above.